# Chatbot Tra Cứu Bộ Luật Dân Sự Việt Nam 2015


In [ ]:

import importlib
checks = {
    "pdf2image": "pdf2image", "pytesseract": "pytesseract", "tqdm": "tqdm",
    "dotenv": "python-dotenv", "sentence_transformers": "sentence-transformers",
    "faiss": "faiss-cpu", "chromadb": "chromadb", "qdrant_client": "qdrant-client",
    "langchain": "langchain", "langgraph": "langgraph",
    "google.generativeai": "google-generativeai", "whisper": "openai-whisper",
    "fastapi": "fastapi", "sounddevice": "sounddevice", "gtts": "gTTS",
}
missing = [pkg for mod, pkg in checks.items() if not importlib.util.find_spec(mod)]
print("Missing:", missing if missing else "none — all good!")


In [ ]:
# BƯỚC 1: Chuẩn bị môi trường
from setup import run_setup
status = run_setup()


In [ ]:
# BƯỚC 2: OCR PDF scan → full_text.txt
from ocr import run_ocr, preview_ocr_result
import re

full_text = run_ocr(force_rerun=False)

preview_ocr_result(n_chars=800)

articles = sorted(set(int(x) for x in re.findall(r'Điều\s+(\d+)\.', full_text)))
print(f"\nTìm thấy {len(articles)} Điều luật — từ Điều {articles[0]} đến Điều {articles[-1]}")


In [ ]:
# BƯỚC 3: Whisper transcribe audio → data/transcripts/transcripts.json
from transcribe import run_transcribe

# model_size: 'base' (nhanh) hoặc 'small' (chính xác hơn)
transcripts = run_transcribe(model_size="base", force_rerun=False)


In [ ]:
# BƯỚC 4: Chunking (fixed-size + theo Điều luật)
from chunking import run_chunking

chunks_fixed, chunks_article = run_chunking(full_text=full_text, transcripts=transcripts, force_rerun=True)


In [ ]:
# BƯỚC 5: Embedding → data/embeddings/
from embedding import run_embedding

embeddings_fixed, embeddings_article = run_embedding(chunks_fixed, chunks_article, force_rerun=True)


In [ ]:
from vectordb import run_vectordb

try: qdrant_cl.close()
except: pass

(faiss_index, faiss_chunks), chroma_col, qdrant_cl = run_vectordb(
    chunks_article, embeddings_article, force_rerun=True
)

In [ ]:
print("FAISS dim:", faiss_index.d)
print("Query vec dim:", embeddings_article.shape[1])

In [ ]:
# BƯỚC 7: Benchmark + bảng so sánh Vector DB & Chunking Strategy
import importlib
import embedding, vectordb, benchmark
importlib.reload(embedding)
importlib.reload(vectordb)
importlib.reload(benchmark)
from benchmark import run_benchmark

df_vectordb, df_chunking = run_benchmark(
    faiss_index=faiss_index, faiss_chunks=faiss_chunks,
    chroma_col=chroma_col, qdrant_cl=qdrant_cl,
    chunks_fixed=chunks_fixed, embeddings_fixed=embeddings_fixed,
    full_text=full_text,
)


In [ ]:
# BƯỚC 8: Giao diện truy vấn đa phương thức (text / mic / ảnh + TTS)
import os
from google import genai
from embedding import embed_texts
from query import query, interactive_loop
from vectordb import search_qdrant

_gemini_client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

class QdrantStore:
    def search(self, question: str, top_k: int = 5):
        q_vec = embed_texts([question])[0]
        return search_qdrant(q_vec, qdrant_cl, top_k=top_k)

class GeminiModel:
    def generate_content(self, prompt: str):
        class _R:
            def __init__(self, text): self.text = text
        resp = _gemini_client.models.generate_content(model="gemini-3-flash-preview", contents=prompt)
        return _R(resp.text)

qdrant_store = QdrantStore()
gemini_model = GeminiModel()

result = query(
    text="Năng lực pháp luật dân sự của cá nhân là gì?",
    vector_store=qdrant_store,
    gemini_model=gemini_model,
)
print("Câu hỏi  :", result["question"])
print("Trả lời  :", result["answer"][:500])


In [ ]:
# BƯỚC 8 (tiếp): Mic / Ảnh
result = query(image_path="data/test_image/1.png",
               vector_store=qdrant_store, gemini_model=gemini_model)


In [ ]:
# # Ghi âm 5 giây từ mic
# result = query(use_mic=True, mic_duration=5,
#                vector_store=qdrant_store, gemini_model=gemini_model,
#                respond_with_audio=True)

In [ ]:
%pip install -U ddgs

import ddgs
# BƯỚC 9: Agent (LangChain ReAct + LangGraph + Tools + Memory)
from vectordb import search_qdrant
from agent import run_agent

def _qdrant_search(q_vec, top_k=5):
    return search_qdrant(q_vec, qdrant_cl, top_k=top_k)

react_executor, graph_app = run_agent(search_fn=_qdrant_search)

In [ ]:
result = react_executor.invoke({"messages": [("human", "Luật hình sự quy định tội trộm cắp thế nào?")]})

# Xem agent đã dùng tool gì
for msg in result["messages"]:
    print(f"[{type(msg).__name__}]", msg.content[:100] if isinstance(msg.content, str) else "...")

# Câu trả lời cuối
content = result["messages"][-1].content
answer = content[0]["text"] if isinstance(content, list) else content
print("\n--- Trả lời ---")
print(answer)


In [ ]:
from evaluation import run_evaluation

try:
    df_retrieval, df_generation = run_evaluation(
        faiss_index=faiss_index, faiss_chunks=faiss_chunks,
        chroma_col=chroma_col, qdrant_cl=qdrant_cl,
        run_generation_eval=True,
    )
except Exception as e:
    if "429" in str(e) or "RESOURCE_EXHAUSTED" in str(e):
        print("Gemini quota exceeded. Fallback: skip generation eval and run retrieval-only.")
        df_retrieval, df_generation = run_evaluation(
            faiss_index=faiss_index, faiss_chunks=faiss_chunks,
            chroma_col=chroma_col, qdrant_cl=qdrant_cl,
            run_generation_eval=False,
        )
    else:
        raise

In [ ]:
# BƯỚC 13: Báo cáo tổng hợp
import pandas as pd
from pathlib import Path

print("=" * 60)
print("  TỔNG HỢP KẾT QUẢ DỰ ÁN RAG BỘ LUẬT DÂN SỰ 2015")
print("=" * 60)

sections = {
    "Dữ liệu": [
        ("PDF", "data/pdf/data1.pdf"),
        ("OCR text", "data/extracted_text/full_text.txt"),
        ("Transcripts", "data/transcripts/transcripts.json"),
        ("Chunks (fixed)", "data/chunks/chunks_fixed.json"),
        ("Chunks (article)", "data/chunks/chunks_article.json"),
        ("Embeddings fixed", "data/embeddings/embeddings_fixed.npy"),
        ("Embeddings article", "data/embeddings/embeddings_article.npy"),
    ],
    "Vector Stores": [
        ("FAISS index", "vector_stores/faiss/index.faiss"),
        ("Chroma DB", "vector_stores/chroma/chroma.sqlite3"),
        ("Qdrant DB", "vector_stores/qdrant"),
    ],
    "Benchmark": [
        ("VectorDB results", "outputs/benchmark/benchmark_vectordb.csv"),
        ("Chunking results", "outputs/benchmark/benchmark_chunking.csv"),
        ("Test set", "outputs/benchmark/test_set.json"),
    ],
    "Evaluation": [
        ("Retrieval report", "outputs/evaluation/eval_retrieval.csv"),
        ("Generation report", "outputs/evaluation/eval_generation.csv"),
        ("Chart", "outputs/evaluation/eval_chart.png"),
    ],
    "Dataset & Models": [
        ("Q&A dataset", "data/dataset/qa_pairs.jsonl"),
        ("LoRA adapter", "models/lora_adapter"),
        ("Colab script", "models/colab_finetune.py"),
    ],
}

for section, files in sections.items():
    print(f"\n  [{section}]")
    for name, path in files:
        p = Path(path)
        exists = "✓" if p.exists() else "✗"
        size = f"  ({p.stat().st_size // 1024:,} KB)" if p.is_file() and p.exists() else ""
        print(f"    {exists} {name:<22} {path}{size}")

# Bảng benchmark nếu có
for csv_file in ["outputs/benchmark/benchmark_vectordb.csv",
                 "outputs/evaluation/eval_retrieval.csv"]:
    p = Path(csv_file)
    if p.exists():
        print(f"\n  [{p.stem}]")
        print(pd.read_csv(p).to_string(index=False))

print("\n" + "=" * 60)
print("  ✓ Dự án hoàn tất!")
print("  API: uvicorn api:app --reload --port 8000")
print("=" * 60)

In [ ]:
# BƯỚC 10: Fine-tuning LoRA — tạo Q&A dataset, hướng dẫn train Colab
from finetune import run_finetune

qa_pairs = run_finetune(chunks_article=chunks_article)